In [ ]:
# =========================================
# STEP 1: IMPORT LIBRARIES
# =========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn - machine learning tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Imbalance handling
from imblearn.over_sampling import SMOTE

# Scaling
from sklearn.preprocessing import StandardScaler

# Save model
import joblib

👉 STEP 1: IMPORT LIBRARIES ONLY

👉 STEP 2: LOAD DATASET + BASIC CHECK

In [ ]:
# =========================================
# STEP 2: LOAD DATASET
# =========================================

# Load CSV file
df = pd.read_csv("heart.csv")

# Show first 5 rows
print("FIRST 5 ROWS:")
print(df.head())

# Show dataset info (columns, types, missing values)
print("\nDATASET INFO:")
print(df.info())

# Show statistical summary
print("\nDATASET DESCRIPTION:")
print(df.describe())

# Check missing values
print("\nMISSING VALUES:")
print(df.isnull().sum())

👉 STEP 3: EXPLORATORY DATA ANALYSIS (EDA)

STEP 3 CODE

In [ ]:
# =========================================
# STEP 3: EXPLORATORY DATA ANALYSIS (EDA)
# =========================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -------------------------
# 1. TARGET DISTRIBUTION
# -------------------------
plt.figure(figsize=(6,4))
sns.countplot(x='Heart Disease', data=df)
plt.title("Target Distribution (Heart Disease)")
plt.show()


# -------------------------
# 2. CORRELATION HEATMAP (FIXED)
# -------------------------
plt.figure(figsize=(12,8))

# only numeric columns (IMPORTANT FIX)
numeric_df = df.select_dtypes(include=[np.number])

sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap (Numeric Only)")
plt.show()


# -------------------------
# 3. AGE DISTRIBUTION (if exists)
# -------------------------
if 'age' in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df['age'], bins=20, kde=True)
    plt.title("Age Distribution")
    plt.show()


# -------------------------
# 4. CHECK DATA TYPES (IMPORTANT DEBUG STEP)
# -------------------------
print("\nCOLUMN TYPES:")
print(df.dtypes)


# -------------------------
# 5. QUICK VALUE CHECK (optional debugging)
# -------------------------
print("\nUNIQUE VALUES IN TARGET:")
print(df['Heart Disease'].unique())

👉 STEP 4: DATA PREPROCESSING (encoding + cleaning + ML-ready dataset)

In [ ]:
# =========================================
# STEP 4: DATA PREPROCESSING (FIXED)
# =========================================

from sklearn.preprocessing import LabelEncoder

# -------------------------
# 1. COPY DATA (SAFE PRACTICE)
# -------------------------
data = df.copy()

# -------------------------
# 2. HANDLE MISSING VALUES
# -------------------------
data.fillna(data.mean(numeric_only=True), inplace=True)

# -------------------------
# 3. ENCODE TEXT COLUMNS
# -------------------------
le = LabelEncoder()

for col in data.columns:
    if data[col].dtype == 'object':
        data[col] = le.fit_transform(data[col])

# -------------------------
# 4. SPLIT FEATURES & TARGET
# -------------------------
X = data.drop('Heart Disease', axis=1)
y = data['Heart Disease']

# -------------------------
# 5. SAVE FEATURE NAMES
# -------------------------
feature_names = X.columns

# -------------------------
# 6. FINAL CHECK
# -------------------------
print("SHAPE OF X:", X.shape)
print("SHAPE OF y:", y.shape)

print("\nFEATURE PREVIEW:")
print(X.head())

print("\nTARGET DISTRIBUTION:")
print(y.value_counts())

Cheack dataset column name

In [ ]:
print(df.columns)

👉 STEP 5: TRAIN-TEST SPLIT + SMOTE + SCALING

In [ ]:
# =========================================
# STEP 5: TRAIN-TEST SPLIT + SMOTE + SCALING
# =========================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# -------------------------
# 1. TRAIN-TEST SPLIT
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# -------------------------
# 2. HANDLE IMBALANCE (SMOTE)
# -------------------------
smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print("X_train shape:", X_train.shape)
print("y_train distribution:")
print(y_train.value_counts())

# -------------------------
# 3. FEATURE SCALING
# -------------------------
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to DataFrame (keep feature names)
X_train = pd.DataFrame(X_train, columns=feature_names)
X_test = pd.DataFrame(X_test, columns=feature_names)

print("\nScaling done successfully ✔")

👉 STEP 6: MODEL TRAINING (Logistic Regression + Random Forest + SVM + XGBoost)

In [ ]:
# =========================================
# STEP 6: CLEAN TARGET FIX + MODEL TRAINING
# =========================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# -------------------------
# 1. FIX TARGET FIRST (VERY IMPORTANT)
# -------------------------
# Convert target BEFORE anything else
y = y.replace({'Absence': 0, 'Presence': 1})

# Make sure it's clean integer type
y = y.astype(int)

# -------------------------
# 2. SPLIT AGAIN (SAFE RESET)
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------
# 3. SMOTE (ONLY ON TRAIN)
# -------------------------
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# -------------------------
# 4. SCALING
# -------------------------
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -------------------------
# 5. MODELS
# -------------------------
lr = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier(n_estimators=200, random_state=42)
svm = SVC(probability=True)
xgb = XGBClassifier(eval_metric='logloss')

# -------------------------
# 6. TRAIN MODELS
# -------------------------
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
svm.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# -------------------------
# 7. PREDICTIONS
# -------------------------
lr_pred = lr.predict(X_test)
rf_pred = rf.predict(X_test)
svm_pred = svm.predict(X_test)
xgb_pred = xgb.predict(X_test)

print("✅ All models trained successfully")

👉 STEP 7: MODEL EVALUATION (Accuracy, ROC-AUC, Confusion Matrix, Best Model Selection)

In [ ]:
# =========================================
# STEP 7: MODEL EVALUATION
# =========================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

# -------------------------
# 1. ACCURACY SCORES
# -------------------------
print("🔹 MODEL ACCURACY SCORES\n")

models = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "SVM": svm,
    "XGBoost": xgb
}

preds = {
    "Logistic Regression": lr_pred,
    "Random Forest": rf_pred,
    "SVM": svm_pred,
    "XGBoost": xgb_pred
}

probs = {
    "Logistic Regression": lr.predict_proba(X_test)[:, 1],
    "Random Forest": rf.predict_proba(X_test)[:, 1],
    "SVM": svm.predict_proba(X_test)[:, 1],
    "XGBoost": xgb.predict_proba(X_test)[:, 1]
}

results = {}

for name in models:
    acc = accuracy_score(y_test, preds[name])
    roc = roc_auc_score(y_test, probs[name])
    
    results[name] = roc  # we use ROC as best metric

    print(f"{name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC-AUC: {roc:.4f}")
    print("-"*30)


# -------------------------
# 2. BEST MODEL SELECTION
# -------------------------
best_model_name = max(results, key=results.get)
print("\n🏆 BEST MODEL:", best_model_name)


# -------------------------
# 3. CONFUSION MATRIX (BEST MODEL)
# -------------------------
best_pred = preds[best_model_name]

cm = confusion_matrix(y_test, best_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()


# -------------------------
# 4. FULL REPORT (BEST MODEL)
# -------------------------
print("\n📊 CLASSIFICATION REPORT:")
print(classification_report(y_test, best_pred))

Awesome 🔥 now we turn your ML project into a real web app using Streamlit

🚀 STEP 1: INSTALL STREAMLIT

In [ ]:
# pip install streamlit joblib

In [ ]:
# pip install streamlit scikit-learn pandas

🚀 STEP 2: CREATE FILE app.py

Run
streamlit run app.py


In [ ]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier

# Load dataset
df = pd.read_csv("heart.csv")  # make sure file exists

# Split features and target
X = df.drop("Heart Disease", axis=1)
y = df["Heart Disease"]

# Train model
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X, y)

# Save model
joblib.dump(model, "heart_model.pkl")

print("Model saved successfully ✔")

In [ ]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("heart.csv")

# fix target
if df["Heart Disease"].dtype == "object":
    df["Heart Disease"] = df["Heart Disease"].map({
        "Yes": 1,
        "No": 0,
        "Presence": 1,
        "Absence": 0
    })

X = df.drop("Heart Disease", axis=1)
y = df["Heart Disease"]

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X, y)

# SAVE BOTH FILES
joblib.dump(model, "heart_model.pkl")
joblib.dump(X.columns, "features.pkl")

print("✅ Model + Features saved successfully")

In [ ]:
import streamlit as st
import pandas as pd
import joblib
import matplotlib.pyplot as plt

# =========================================
# PAGE CONFIG
# =========================================
st.set_page_config(
    page_title="Hospital AI System",
    page_icon="🏥",
    layout="wide"
)

# =========================================
# LOAD MODEL
# =========================================
model = joblib.load("heart_model.pkl")
features = joblib.load("features.pkl")

# =========================================
# CSS DESIGN (HOSPITAL STYLE)
# =========================================
st.markdown("""
<style>

body {
    background-color: #f5f7fb;
}

.main-title {
    font-size: 42px;
    font-weight: 700;
    color: #0b3d91;
    text-align: center;
}

.sub-title {
    text-align: center;
    color: gray;
    font-size: 16px;
}

.card {
    background: white;
    padding: 20px;
    border-radius: 15px;
    box-shadow: 0px 4px 20px rgba(0,0,0,0.08);
}

.risk-high {
    padding: 15px;
    background-color: #ffebee;
    color: #c62828;
    border-radius: 10px;
    font-weight: bold;
    text-align: center;
}

.risk-low {
    padding: 15px;
    background-color: #e8f5e9;
    color: #2e7d32;
    border-radius: 10px;
    font-weight: bold;
    text-align: center;
}

</style>
""", unsafe_allow_html=True)

# =========================================
# HEADER
# =========================================
st.markdown("<div class='main-title'>🏥 Hospital AI Diagnostic System</div>", unsafe_allow_html=True)
st.markdown("<div class='sub-title'>Clinical Decision Support for Heart Disease Prediction</div>", unsafe_allow_html=True)

st.write("---")

# =========================================
# LAYOUT
# =========================================
col1, col2 = st.columns(2)

with col1:
    st.markdown("### 🧍 Patient Clinical Data")

    Age = st.slider("Age", 20, 80, 40)
    Sex = st.selectbox("Sex (0 = Female, 1 = Male)", [0, 1])
    ChestPain = st.slider("Chest Pain Type", 0, 3, 1)
    BP = st.slider("Blood Pressure (mmHg)", 80, 200, 120)
    Chol = st.slider("Cholesterol (mg/dL)", 100, 400, 200)
    FBS = st.selectbox("Fasting Blood Sugar > 120", [0, 1])

with col2:
    st.markdown("### ❤️ Cardiac Test Results")

    EKG = st.slider("ECG Results", 0, 2, 1)
    MaxHR = st.slider("Max Heart Rate", 60, 200, 150)
    ExerciseAngina = st.selectbox("Exercise Angina", [0, 1])
    STDep = st.slider("ST Depression", 0.0, 6.0, 1.0)
    Slope = st.slider("ST Slope", 0, 2, 1)
    Vessels = st.slider("Fluoroscopy Vessels", 0, 3, 0)
    Thallium = st.slider("Thallium Stress Test", 0, 3, 2)

st.write("---")

# =========================================
# INPUT DATA
# =========================================
input_df = pd.DataFrame([[
    Age, Sex, ChestPain, BP, Chol, FBS,
    EKG, MaxHR, ExerciseAngina, STDep,
    Slope, Vessels, Thallium
]], columns=features)

# =========================================
# PREDICTION
# =========================================
if st.button("🩺 Run Clinical Diagnosis"):

    prediction = model.predict(input_df)[0]
    proba = model.predict_proba(input_df)[0]

    st.write("---")

    # =========================================
    # RESULT DISPLAY (HOSPITAL STYLE)
    # =========================================
    if prediction == 1:
        st.markdown("<div class='risk-high'>⚠ HIGH RISK OF HEART DISEASE</div>", unsafe_allow_html=True)
    else:
        st.markdown("<div class='risk-low'>✅ LOW RISK OF HEART DISEASE</div>", unsafe_allow_html=True)

    # =========================================
    # PROBABILITY
    # =========================================
    st.subheader("📊 Risk Probability")
    st.write({
        "No Disease": round(proba[0], 3),
        "Disease": round(proba[1], 3)
    })

    st.progress(float(proba[1]))

    # =========================================
    # AI EXPLANATION
    # =========================================
    st.write("---")
    st.subheader("🧠 Clinical AI Explanation")

    importance = model.feature_importances_

    exp_df = pd.DataFrame({
        "Feature": features,
        "Impact Score": importance
    }).sort_values("Impact Score")

    fig, ax = plt.subplots()
    ax.barh(exp_df["Feature"], exp_df["Impact Score"])
    ax.set_title("Key Medical Risk Factors")

    st.pyplot(fig)

    st.write("### 🔥 Top Risk Contributors")
    st.write(exp_df.sort_values("Impact Score", ascending=False).head(3))

INSTALL SHAP

In [ ]:
# pip install shap

In [ ]:
st.write("---")
st.subheader("🧠 AI Explainability (Why Prediction Happened)")

import numpy as np
import matplotlib.pyplot as plt

# =========================
# GET SHAP VALUES
# =========================
shap_values = explainer.shap_values(input_df)

# binary classification fix
if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap_values = np.array(shap_values)

# =========================
# SAFELY FLATTEN
# =========================
shap_row = shap_values[0].flatten()

# =========================
# FORCE FEATURE MATCHING
# =========================
feature_names = input_df.columns.tolist()

min_len = min(len(feature_names), len(shap_row))

feature_names = feature_names[:min_len]
shap_row = shap_row[:min_len]

# =========================
# PLOT
# =========================
fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(feature_names, shap_row)
ax.set_xlabel("Impact on Prediction")
ax.set_title("Feature Contribution (SHAP Explanation)")

st.pyplot(fig)

# =========================
# TOP FEATURES
# =========================
st.write("### 🔥 Key Risk Factors")

top_idx = np.argsort(np.abs(shap_row))[::-1][:3]

for i in top_idx:
    st.write(f"• **{feature_names[i]}** → impact: {round(shap_row[i], 4)}")

BEAUTIFUL UI UPGRADE

In [ ]:
import streamlit as st

# =========================
# PAGE CONFIG
# =========================
st.set_page_config(
    page_title="AI Hospital Dashboard",
    page_icon="🏥",
    layout="wide"
)

# =========================
# CUSTOM CSS (HOSPITAL UI STYLE)
# =========================
st.markdown("""
<style>

body {
    background-color: #f4f8fb;
}

.main-title {
    font-size: 40px;
    font-weight: 800;
    color: #0b3d91;
    text-align: center;
}

.sub-title {
    font-size: 18px;
    color: #4b5563;
    text-align: center;
    margin-bottom: 20px;
}

.card {
    background: white;
    padding: 20px;
    border-radius: 15px;
    box-shadow: 0px 4px 20px rgba(0,0,0,0.08);
    margin-bottom: 15px;
}

.result-safe {
    background-color: #d1fae5;
    padding: 15px;
    border-radius: 10px;
    color: #065f46;
    font-weight: bold;
}

.result-risk {
    background-color: #fee2e2;
    padding: 15px;
    border-radius: 10px;
    color: #991b1b;
    font-weight: bold;
}

</style>
""", unsafe_allow_html=True)

# =========================
# HEADER
# =========================
st.markdown('<div class="main-title">🏥 AI Hospital Diagnosis System</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-title">Advanced Machine Learning + SHAP Explainability Dashboard</div>', unsafe_allow_html=True)

st.write("---")

IMPROVED INPUT SECTION (CLEAN UI)

In [ ]:
col1, col2, col3 = st.columns(3)

with col1:
    st.markdown("### 👤 Patient Info")
    Age = st.slider("Age", 20, 80, 40)
    Sex = st.selectbox("Sex", ["Male", "Female"])
    Sex = 1 if Sex == "Male" else 0

    ChestPain = st.slider("Chest Pain Type", 0, 3, 1)

with col2:
    st.markdown("### ❤️ Health Metrics")
    BP = st.slider("Blood Pressure", 80, 200, 120)
    Chol = st.slider("Cholesterol", 100, 400, 200)
    MaxHR = st.slider("Max Heart Rate", 60, 200, 150)

with col3:
    st.markdown("### 🧪 Medical Tests")
    FBS = st.selectbox("FBS > 120", [0, 1])
    EKG = st.slider("EKG Results", 0, 2, 1)
    ExerciseAngina = st.selectbox("Exercise Angina", [0, 1])
    STDep = st.slider("ST Depression", 0.0, 6.0, 1.0)
    Slope = st.slider("Slope", 0, 2, 1)
    Vessels = st.slider("Number of vessels", 0, 3, 0)
    Thallium = st.slider("Thallium", 0, 3, 2)

🩺 RESULT UI (VERY CLEAN)

In [ ]:
if st.button("🧠 Analyze Patient"):

    st.write("---")

    if prediction == 1:
        st.markdown('<div class="result-risk">⚠ HIGH RISK DETECTED</div>', unsafe_allow_html=True)
    else:
        st.markdown('<div class="result-safe">✅ LOW RISK DETECTED</div>', unsafe_allow_html=True)

    st.progress(float(proba[1]))

    st.write("### 📊 Probability")
    st.json({
        "No Disease": float(proba[0]),
        "Disease": float(proba[1])
    })

STEP 2 — SAVE PATIENT DATA AFTER PREDICTION

🤖🏥 GPT-LEVEL MEDICAL CHATBOT (FULL SETUP)
STEP 1 — INSTALL LIBRARY

In [ ]:
# pip install openai

🔵 STEP 2 — SET API KEY (IMPORTANT)
[API of chatgpt](https://platform.openai.com/api-keys)

🔵 STEP 3 — IMPORT GPT CLIENT

In [ ]:
# pip install python-dotenv

In [ ]:
# pip install openai streamlit python-dotenv

FULL PDF MEDICAL REPORT SYSTEM
🔵 STEP 1 — INSTALL LIBRARY

In [ ]:
# pip install reportlab